In [ ]:
import duckdb
import pandas as pd

school_code = 302817872
school_name = "Vilniaus Antakalnio gimnazija"
year = 2024

with duckdb.connect(
    "../data/local/lithuanian_education_warehouse.duckdb",
    read_only=True,
) as connection:
    school_funding = connection.execute(
        """
        SELECT *
        FROM finansavimas_mokykla_metai
        WHERE mokyklos_kodas = ?
    """,
        [school_code],
    ).df()

    school_results = connection.execute(
        """
        SELECT
            YEAR(ataskaitinis_periodas) AS metai,
            saskaita_lygis_1,
            kodas_lygis_2,
            saskaita_lygis_2,
            saskaita_lygis_3,
            CAST(suma AS DOUBLE) AS suma
        FROM mokyklu_veiklos_rezultatai
        WHERE mokyklos_kodas = ?
          AND saskaita_lygis_1 IN ('Pajamos', 'Sąnaudos')
    """,
        [school_code],
    ).df()

selected_results = school_results[school_results["metai"] == year].copy()

display(selected_results.head())

,metai,saskaita_lygis_1,kodas_lygis_2,saskaita_lygis_2,saskaita_lygis_3,suma
175,2024,Pajamos,1.1.,Finansavimo pajamos,Iš savivaldybių biudžetų,8.725113e+05
176,2024,Pajamos,1.1.,Finansavimo pajamos,Iš valstybės biudžeto,2.364120e+06
177,2024,Pajamos,1.3.,Pagrindinės veiklos ir kitos pajamos,Pagrindinės veiklos kitos pajamos,1.698863e+04
178,2024,Pajamos,1.1.,Finansavimo pajamos,Iš kitų finansavimo šaltinių,3.273515e+04
179,2024,Pajamos,1.1.,Finansavimo pajamos,"Iš ES, užsienio valstybių ir tarptautinių orga...",6.805050e+04


In [ ]:
income = selected_results[selected_results["saskaita_lygis_1"] == "Pajamos"]

expenses = selected_results[selected_results["saskaita_lygis_1"] == "Sąnaudos"]

funding_income = (
    school_funding.loc[school_funding["metai"] == year, "suma"]
    .astype("float64")
    .sum(min_count=1)
)

other_income_rows = income.loc[
    income["saskaita_lygis_2"] != "Finansavimo pajamos",
    "suma",
]

other_income = 0.0 if other_income_rows.empty else other_income_rows.sum(min_count=1)

total_income = income["suma"].sum(min_count=1)
total_expenditure = -expenses["suma"].sum(min_count=1)

figures = {
    "Funding income": funding_income,
    "Other reported income": other_income if pd.notna(total_income) else None,
    "Total reported income": total_income,
    "Total reported expenditure": total_expenditure,
}

print(f"{school_name} — {year}")

for label, amount in figures.items():
    displayed_amount = f"€{amount:,.2f}" if pd.notna(amount) else "Unavailable"
    print(f"{label}: {displayed_amount}")

Vilniaus Antakalnio gimnazija — 2024
Funding income: €3,337,417.21
Other reported income: €16,988.63
Total reported income: €3,354,405.84
Total reported expenditure: €3,366,732.04


### School Income and Expenditure - Vilniaus Antakalnio Gimnazija, 2024

This block uses `finansavimas_mokykla_metai` for funding income and
`mokyklu_veiklos_rezultatai` for the school’s reported income and expenditure.

Funding income is added across all recorded funding sources. Other reported
income includes income categories outside “Finansavimo pajamos”. Total reported
income combines the income entries in the financial-results table.

Total expenditure is calculated by adding the expense entries and reversing
their sign, following the dashboard’s calculation. Amounts are shown in euros
to two decimal places, with missing totals displayed as unavailable.

In [ ]:
funding_by_year = (
    school_funding.assign(suma=school_funding["suma"].astype("float64"))
    .groupby("metai")["suma"]
    .sum(min_count=1)
    .rename("Funding received")
)

expenditure_by_year = (
    school_results[school_results["saskaita_lygis_1"] == "Sąnaudos"]
    .groupby("metai")["suma"]
    .sum(min_count=1)
    .mul(-1)
    .rename("Total expenditure")
)

financial_history = pd.concat(
    [funding_by_year, expenditure_by_year],
    axis=1,
).sort_index()

financial_history = (
    financial_history.reindex(
        range(
            int(financial_history.index.min()),
            int(financial_history.index.max()) + 1,
        )
    )
    .rename_axis("Year")
    .reset_index()
)

display(financial_history)

,Year,Funding received,Total expenditure
0,2015,9.681749e+05,9.746851e+05
1,2016,9.611868e+05,9.743190e+05
2,2017,1.083172e+06,1.086911e+06
3,2018,1.069625e+06,1.083663e+06
4,2019,1.275204e+06,1.285189e+06
5,2020,1.375281e+06,1.375972e+06
6,2021,1.727423e+06,1.738228e+06
7,2022,2.259036e+06,2.270427e+06
8,2023,2.619401e+06,2.632475e+06
9,2024,3.337417e+06,3.366732e+06


In [4]:
import plotly.express as px

figure = px.line(
    financial_history,
    x="Year",
    y=["Funding received", "Total expenditure"],
    markers=True,
    color_discrete_map={
        "Funding received": "#2563eb",
        "Total expenditure": "#f97316",
    },
    labels={
        "value": "Amount (€)",
        "variable": "",
    },
)

figure.update_traces(
    connectgaps=False,
    line=dict(width=3),
    marker=dict(size=7),
)

figure.update_layout(
    title=f"Funding received and expenditure over time — {school_name}",
    height=450,
    margin=dict(l=20, r=20, t=70, b=40),
    legend=dict(
        orientation="h",
        y=1.12,
        x=0,
    ),
)

figure.update_xaxes(dtick=1, tickformat="d")

figure.update_yaxes(
    tickprefix="€",
    tickformat="",
    exponentformat="B",
)

figure.show(config={"staticPlot": True, "displayModeBar": False})

### Funding Received and Expenditure Over Time

This chart groups the school’s recorded funding and expenditure by calendar
year. Funding comes from `finansavimas_mokykla_metai`, while expenditure comes
from `mokyklu_veiklos_rezultatai`, with expense signs reversed.

The blue line shows funding received, and the orange line shows total
expenditure. Each point represents one year, with amounts expressed in
nominal euros. Missing figures appear as gaps.

The gap between the lines compares funding income with expenditure.
The school may also have other sources of income.

In [5]:
expense_lines = expenses.dropna(subset=["suma"]).copy()

expense_lines["expenditure_eur"] = -expense_lines["suma"]

expenditure_by_category = (
    expense_lines.groupby(
        ["kodas_lygis_2", "saskaita_lygis_2"],
        as_index=False,
    )["expenditure_eur"]
    .sum()
    .sort_values("expenditure_eur", ascending=False)
)

display(expenditure_by_category)

,kodas_lygis_2,saskaita_lygis_2,expenditure_eur
0,2.1.,Pagrindinės veiklos sąnaudos,3.366720e+06
1,2.3.,Finansinės ir investicinės veiklos sąnaudos,1.251000e+01


In [ ]:
figure = px.bar(
    expenditure_by_category,
    x="expenditure_eur",
    y="saskaita_lygis_2",
    orientation="h",
    text=[f"€{amount:,.0f}" for amount in expenditure_by_category["expenditure_eur"]],
    labels={
        "expenditure_eur": "Reported expenditure (€)",
        "saskaita_lygis_2": "Expense category",
    },
    color_discrete_sequence=["#2563eb"],
)

figure.update_traces(
    textposition="outside",
    cliponaxis=False,
)

figure.update_layout(
    title=f"Reported expenditure by category — {school_name}, {year}",
    height=400,
    margin=dict(l=20, r=120, t=70, b=40),
)

figure.update_xaxes(
    tickprefix="€",
    tickformat="",
    exponentformat="B",
)

figure.update_yaxes(
    categoryorder="array",
    categoryarray=expenditure_by_category["saskaita_lygis_2"].tolist(),
    autorange="reversed",
    automargin=True,
    title=None,
)

figure.show(config={"staticPlot": True, "displayModeBar": False})

### Reported Expenditure by Category - 2024

This chart uses the selected school’s expense records from
`mokyklu_veiklos_rezultatai`, grouped by their level-2 category codes and names.

Expense amounts are added within each category and their signs are reversed,
following the dashboard’s calculation.

The horizontal bars show expenditure from largest to smallest, with amounts
labelled in euros rounded to the nearest euro. Records with missing amounts
are excluded.

In [ ]:
selected_category = "Pagrindinės veiklos sąnaudos"

main_operating_expenses = expense_lines[
    expense_lines["saskaita_lygis_2"] == selected_category
].copy()

operating_expenses_by_type = (
    main_operating_expenses.groupby(
        "saskaita_lygis_3",
        as_index=False,
    )["expenditure_eur"]
    .sum()
    .sort_values("expenditure_eur", ascending=False)
)

display(operating_expenses_by_type)

In [ ]:
figure = px.bar(
    operating_expenses_by_type,
    x="expenditure_eur",
    y="saskaita_lygis_3",
    orientation="h",
    text=[
        f"€{amount:,.0f}" for amount in operating_expenses_by_type["expenditure_eur"]
    ],
    labels={
        "expenditure_eur": "Reported expenditure (€)",
        "saskaita_lygis_3": "Expense type",
    },
    color_discrete_sequence=["#2563eb"],
)

figure.update_traces(
    textposition="outside",
    cliponaxis=False,
)

figure.update_layout(
    title=f"{selected_category} - {school_name}, {year}",
    height=max(450, 100 + 45 * len(operating_expenses_by_type)),
    margin=dict(l=20, r=120, t=70, b=40),
)

figure.update_xaxes(
    tickprefix="€",
    tickformat="",
    exponentformat="B",
)

figure.update_yaxes(
    categoryorder="array",
    categoryarray=operating_expenses_by_type["saskaita_lygis_3"].tolist(),
    autorange="reversed",
    automargin=True,
    title=None,
)

figure.show(config={"staticPlot": True, "displayModeBar": False})

### Main Operating Expenditure - 2024

This chart selects the school’s expense records classified as
“Pagrindinės veiklos sąnaudos” and groups them by their level-3 expense types.

The horizontal bars show how expenditure within this broad category is
distributed across the detailed expense types, ordered from largest to smallest.
Labels show amounts in euros rounded to the nearest euro.

The calculation uses the same records and sign reversal as the previous
category chart.

In [10]:
selected_category = "Finansinės ir investicinės veiklos sąnaudos"

financial_investment_expenses = expense_lines[
    expense_lines["saskaita_lygis_2"] == selected_category
].copy()

financial_investment_expenses_by_type = (
    financial_investment_expenses.groupby(
        "saskaita_lygis_3",
        as_index=False,
    )["expenditure_eur"]
    .sum()
    .sort_values("expenditure_eur", ascending=False)
)

display(financial_investment_expenses_by_type)

,saskaita_lygis_3,expenditure_eur
0,Baudų ir delspinigių sąnaudos,12.51


In [ ]:
figure = px.bar(
    financial_investment_expenses_by_type,
    x="expenditure_eur",
    y="saskaita_lygis_3",
    orientation="h",
    text=[
        f"€{amount:,.2f}"
        for amount in financial_investment_expenses_by_type["expenditure_eur"]
    ],
    labels={
        "expenditure_eur": "Reported expenditure (€)",
        "saskaita_lygis_3": "Expense type",
    },
    color_discrete_sequence=["#2563eb"],
)

figure.update_traces(
    textposition="outside",
    cliponaxis=False,
)

figure.update_layout(
    title=f"{selected_category} - {school_name}, {year}",
    height=300,
    margin=dict(l=20, r=100, t=80, b=40),
)

figure.update_xaxes(
    tickprefix="€",
    tickformat=",.2f",
)

figure.update_yaxes(
    categoryorder="array",
    categoryarray=financial_investment_expenses_by_type["saskaita_lygis_3"].tolist(),
    autorange="reversed",
    automargin=True,
    title=None,
)

figure.show(config={"staticPlot": True, "displayModeBar": False})

### Financial and Investment Activity Expenditure - 2024

This chart selects the school’s expense records classified as
“Finansinės ir investicinės veiklos sąnaudos” and groups them by level-3
expense type.

For Vilniaus Antakalnio gimnazija in 2024, the category contains fines and
late-payment charges (“Baudų ir delspinigių sąnaudos”), totalling €12.51.

The bar shows the recorded expenditure after the same sign reversal used
in the previous charts. Amounts are labelled to two decimal places.